# Lab 3 — Python, NumPy, and the Growth Model End to End

**ECON 282E · Session 3 · October 8, 2026**

This lab is Session 3, run by you. By the end you will have solved the optimal growth model
two different ways, graded both against a known answer, and measured what every design
choice cost.

The model throughout is the one from the lectures:

$$v(k,z)=\max_{k'}\{\ln(zk^{\alpha}-k') + \beta\,\mathbb{E}[v(k',z')]\},\qquad \delta=1,$$

which has the **Brock–Mirman closed form**

$$k'(k,z)=\alpha\beta z k^{\alpha},\qquad c(k,z)=(1-\alpha\beta)zk^{\alpha}.$$

Full depreciation is not realistic. It is here because we know the answer exactly, and a
method that cannot reproduce a known answer should not be trusted on an unknown one.

**Prerequisite:** L00. If its check cell did not pass, fix that first.

In [ ]:
import time
import numpy as np
from scipy.optimize import brentq
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
print("numpy", np.__version__)

---
## Part 1 — Python essentials, in five cells

Only the pieces the rest of the lab needs.

In [ ]:
# Containers. A dict is how you carry a calibration around.
par = {"alpha": 0.36, "beta": 0.95, "n": 200, "k_lo": 0.05, "k_hi": 0.60}
print(par["alpha"], len(par))

for key, value in par.items():
    print(f"  {key:6s} = {value}")

In [ ]:
# Functions: default arguments put the calibration in one visible place.
def output(k, z=1.0, alpha=0.36):
    """y = z * k**alpha."""
    return z * k**alpha

print(output(2.0))
print(output(2.0, alpha=0.40))   # name the argument you are changing

In [ ]:
# Returning several things at once, and unpacking them.
def summarise(x):
    return x.min(), x.max(), x.mean()

lo, hi, avg = summarise(np.array([1.0, 2.0, 3.0]))
print(lo, hi, avg)

In [ ]:
# The trap from L00, now in NumPy: a slice is a VIEW, not a copy.
V = np.zeros(5)
part = V[1:3]
part[:] = 7.0
print("V =", V)          # V changed

part = V[1:3].copy()
part[:] = 0.0
print("V =", V)          # V did not change

**Why this matters here.** A value-function loop compares the new guess with the old one.
If `V_new` is a *view* of `V`, the comparison is between an array and itself, the distance
is always zero, and the loop stops after one iteration with a completely wrong answer — and
does not raise an error.

---
## Part 2 — NumPy and the one idea that makes this fast

The Bellman update asks for *every state against every choice*. Written as a double Python
loop that is unusably slow; written as one array expression it is the same arithmetic.

In [ ]:
alpha, beta = 0.36, 0.95
N = 500
k = np.linspace(0.05, 0.60, N)
y = k**alpha
V = np.zeros(N)

# --- the loop you would say out loud ---------------------------------------
t0 = time.perf_counter()
Vn_loop = np.empty(N)
for i in range(N):
    best = -1e18
    for j in range(N):
        c = y[i] - k[j]
        if c > 0:
            val = np.log(c) + beta * V[j]
            if val > best:
                best = val
    Vn_loop[i] = best
t_loop = time.perf_counter() - t0

# --- the same thing, broadcast ---------------------------------------------
t0 = time.perf_counter()
C = y[:, None] - k[None, :]                 # (N, N): C[i, j] = y_i - k_j
U = np.full_like(C, -1e18)
np.log(C, out=U, where=C > 0)
Vn_vec = (U + beta * V[None, :]).max(axis=1)
t_vec = time.perf_counter() - t0

print(f"loop      {t_loop:8.4f} s")
print(f"broadcast {t_vec:8.4f} s")
print(f"speed-up  {t_loop/t_vec:8.1f}x")
print(f"max |difference| = {np.max(np.abs(Vn_loop - Vn_vec)):.1e}")

**Broadcasting, in one rule.** Reading shapes from the right, two arrays are compatible if
each pair of dimensions is equal or one of them is 1; the size-1 dimension is stretched,
without ever being stored.

```
k[:, None]   ->  (N, 1)     rows    = today's state
k[None, :]   ->  (1, N)     columns = tomorrow's choice
difference   ->  (N, N)
```

**Exercise 3.** Broadcasting almost never raises an error — a wrong axis gives you a
plausible array of the wrong shape. Change `axis=1` to `axis=0` in the cell above and look
at what comes out. What economic object did you just compute, if any?

---
## Part 3 — The model as a class

State (parameters, grid, payoff matrix) and behaviour (one Bellman step, solve, the exact
answer) belong together.

In [ ]:
class GrowthModel:
    """u(c) = ln c,  y = z k^alpha,  delta = 1.

    With z omitted the model is deterministic; pass z and Pi for the Markov version.
    """

    def __init__(self, alpha=0.36, beta=0.95, n=200, k_lo=0.05, k_hi=0.60,
                 z=None, Pi=None):
        self.alpha, self.beta = alpha, beta
        self.k = np.linspace(k_lo, k_hi, n)
        self.z = np.array([1.0]) if z is None else np.asarray(z)
        self.Pi = np.array([[1.0]]) if Pi is None else np.asarray(Pi)

        y = self.z[None, :] * (self.k**alpha)[:, None]       # (n, nz)
        C = y[:, :, None] - self.k[None, None, :]            # (n, nz, n)
        self.U = np.full_like(C, -1e18)
        np.log(C, out=self.U, where=C > 0)

    # --- one Bellman step --------------------------------------------------
    def bellman(self, V):
        EV = V @ self.Pi.T                       # EV[l, j] = E[V(k_l, z') | z_j]
        M = self.U + self.beta * EV.T[None, :, :]
        return M.max(axis=2), M.argmax(axis=2)

    def solve(self, tol=1e-6, max_iter=20_000):
        V = np.zeros((self.k.size, self.z.size))
        for n in range(1, max_iter + 1):
            Vn, pol = self.bellman(V)
            d = np.max(np.abs(Vn - V))
            V = Vn
            if d < tol:
                return V, self.k[pol], n
        raise RuntimeError("did not converge")

    # --- the truth ---------------------------------------------------------
    def exact_policy(self):
        return self.alpha * self.beta * self.z[None, :] * self.k[:, None]**self.alpha

    def exact_value(self):
        B = self.alpha / (1 - self.alpha * self.beta)
        b = (np.log(1 - self.alpha * self.beta)
             + (1 + self.beta * B) * np.log(self.z)
             + self.beta * B * np.log(self.alpha * self.beta))
        A = np.linalg.solve(np.eye(self.z.size) - self.beta * self.Pi, b)
        return A[None, :] + B * np.log(self.k)[:, None]

In [ ]:
m = GrowthModel()
t0 = time.perf_counter()
V, g, n = m.solve()
elapsed = time.perf_counter() - t0

pol_err = np.max(np.abs(g / m.exact_policy() - 1))
val_err = np.max(np.abs(V - m.exact_value()))

print(f"iterations           {n}")
print(f"wall time            {elapsed:.3f} s")
print(f"max value error      {val_err:.3e}")
print(f"max rel policy error {pol_err:.3e}")

You should get **272 iterations** and a policy that is wrong by about **1%**.

That 1% is not noise and it will not go away with more iterations. The next cell shows why.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(m.k, g[:, 0], color="firebrick", lw=1.5, label="VFI on the grid")
ax[0].plot(m.k, m.exact_policy()[:, 0], "k--", lw=1.2, label=r"$\alpha\beta k^{\alpha}$")
ax[0].set_xlabel("$k$"); ax[0].set_ylabel("$k'$"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title("the whole grid: they agree")

w = slice(60, 96)
ax[1].step(m.k[w], g[w, 0], where="post", color="firebrick", lw=1.5, label="VFI on the grid")
ax[1].plot(m.k[w], m.exact_policy()[w, 0], "k--", lw=1.2, label=r"$\alpha\beta k^{\alpha}$")
ax[1].set_xlabel("$k$"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[1].set_title("36 consecutive grid points: a staircase")

plt.tight_layout(); plt.show()

print("distinct policy values across those 36 states:", len(np.unique(g[w, 0])))

`argmax` can only return a **grid point**, so the policy is quantised. The error is a
sawtooth of amplitude about half a grid spacing — a property of *how we made the choice*,
not of how long we iterated.

**Exercise 4.** Rerun with `n=800`. Does the policy error fall by a factor of 4, or 2?
Which of the two does the theory predict, and why?

---
## Part 4 — How wrong are you, really?

Three diagnostics. The second one is the one people get wrong.

In [ ]:
# (a) The iteration count belongs to beta, not to the grid.
print(f"{'beta':>6} {'ln(eps)/ln(beta)':>18} {'measured':>10} {'beta/(1-beta)':>15}")
for b in (0.90, 0.95, 0.99):
    _, _, n_b = GrowthModel(beta=b).solve()
    print(f"{b:6.2f} {np.log(1e-6)/np.log(b):18.1f} {n_b:10d} {b/(1-b):15.1f}")

In [ ]:
# (b) The stopping rule is NOT the error.
#     Iterate to machine precision first, to get the discrete fixed point V^h.
m = GrowthModel()
Vh, _, _ = m.solve(tol=1e-14)

V = np.zeros((m.k.size, m.z.size))
rows = []
for n in range(1, 400):
    Vn, _ = m.bellman(V)
    step = np.max(np.abs(Vn - V))
    V = Vn
    rows.append((n, step, np.max(np.abs(V - Vh))))
    if step < 1e-6:
        break

n, step, true_err = rows[-1]
print(f"stopped at iteration {n}")
print(f"  last step  ||V_n - V_(n-1)||  = {step:.3e}")
print(f"  bound  beta/(1-beta) * step   = {m.beta/(1-m.beta)*step:.3e}")
print(f"  TRUE error ||V_n - V^h||      = {true_err:.3e}")
print(f"  ratio bound/truth             = {m.beta/(1-m.beta)*step/true_err:.6f}")

Ask for $10^{-6}$ and you get $1.8\times10^{-5}$. At $\beta=0.95$ the factor
$\beta/(1-\beta)$ is **19**, and here it is not a pessimistic bound — it is *attained*.

In [ ]:
ns   = np.array([r[0] for r in rows])
step = np.array([r[1] for r in rows])
err  = np.array([r[2] for r in rows])

plt.figure(figsize=(7, 4))
plt.plot(ns, np.log10(err), color="firebrick", lw=2, label=r"true error $\|V_n-V^h\|$")
plt.plot(ns, np.log10(step), color="darkorange", ls="--", lw=2,
         label=r"what you monitor $\|V_n-V_{n-1}\|$")
plt.axhline(-6, color="grey", ls=":", label=r"$\varepsilon=10^{-6}$")
plt.xlabel("iteration $n$"); plt.ylabel(r"$\log_{10}$ error")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("vertical gap between the lines:", np.mean(np.log10(err) - np.log10(step)).round(4))
print("log10(19) =", round(float(np.log10(19)), 4))

In [ ]:
# (c) Error budgets compose: refining the grid stops helping once the
#     iteration tolerance puts a floor underneath it.
print(f"{'N':>6} {'h':>10} {'value err (1e-6)':>18} {'value err (1e-11)':>19} {'policy err':>12}")
for N in (50, 100, 200, 400, 800):
    loose = GrowthModel(n=N); Vl, gl, _ = loose.solve(tol=1e-6)
    tight = GrowthModel(n=N); Vt, gt, _ = tight.solve(tol=1e-11)
    h = loose.k[1] - loose.k[0]
    print(f"{N:6d} {h:10.2e} "
          f"{np.max(np.abs(Vl-loose.exact_value())):18.3e} "
          f"{np.max(np.abs(Vt-tight.exact_value())):19.3e} "
          f"{np.max(np.abs(gl-loose.exact_policy())):12.3e}")

print(f"\ntruncation floor beta/(1-beta) * 1e-6 = {0.95/0.05*1e-6:.2e}")

Read the last two columns. At `tol=1e-6` the value error stalls around $1.8\times10^{-5}$
— the floor — while at `tol=1e-11` it keeps falling. **Refining a grid while the tolerance
is loose buys nothing**, and you would never see that without running both.

---
## Part 5 — The other road: time iteration

Instead of maximising the value, solve the Euler equation for the policy, one scalar
root-find per state. `brentq` is a black box today — but a licensed one: the residual is
increasing on one side and decreasing on the other, so the root exists and is unique.

In [ ]:
z = np.array([0.95, 1.05])
Pi = np.array([[0.90, 0.10], [0.10, 0.90]])
ms = GrowthModel(z=z, Pi=Pi)
alpha, beta, k = ms.alpha, ms.beta, ms.k
nz = z.size


def time_iteration(model, tol=1e-8, max_iter=1000):
    k, z, Pi = model.k, model.z, model.Pi
    alpha, beta = model.alpha, model.beta
    y = z[None, :] * (k**alpha)[:, None]
    c = 0.5 * y                                   # guess: save half
    n_roots = 0

    for n in range(1, max_iter + 1):
        c_new = np.empty_like(c)
        for j in range(z.size):
            for i in range(k.size):
                yi = y[i, j]

                def resid(cc, yi=yi, j=j, c=c):
                    kp = yi - cc
                    rhs = sum(Pi[j, mm] * alpha * z[mm] * kp**(alpha - 1)
                              / np.interp(kp, k, c[:, mm]) for mm in range(z.size))
                    return 1.0 / cc - beta * rhs

                c_new[i, j] = brentq(resid, 1e-10, yi - 1e-10, xtol=1e-14, rtol=1e-14)
                n_roots += 1
        d = np.max(np.abs(c_new - c))
        c = c_new
        if d < tol:
            return c, n, n_roots
    raise RuntimeError("did not converge")


t0 = time.perf_counter(); c_ti, n_ti, n_roots = time_iteration(ms); t_ti = time.perf_counter() - t0
t0 = time.perf_counter(); Vs, g_vfi, n_vfi = ms.solve();             t_vfi = time.perf_counter() - t0

c_exact = (1 - alpha * beta) * z[None, :] * k[:, None]**alpha
c_vfi = z[None, :] * (k**alpha)[:, None] - g_vfi

print(f"{'':22s}{'VFI':>12}{'time iteration':>18}")
print(f"{'iterations':22s}{n_vfi:12d}{n_ti:18d}")
print(f"{'wall time (s)':22s}{t_vfi:12.3f}{t_ti:18.3f}")
print(f"{'root-finds':22s}{0:12d}{n_roots:18d}")
print(f"{'max rel policy error':22s}"
      f"{np.max(np.abs(c_vfi/c_exact - 1)):12.2e}{np.max(np.abs(c_ti/c_exact - 1)):18.2e}")

Sixteen times fewer iterations, and about a **thousand times** more accurate — at the cost of
6,800 root-finds in pure Python, which is why the wall time goes the other way. Session 4
makes that inner loop cheap.

---
## Part 6 — Grade both, on points neither one solved

The **unit-free Euler residual**: at a capital level *off* the solution grid, compute the
consumption the Euler equation asks for and compare it with the one the method reports.

$$\mathcal{E}(k,z)=\left|1-\tilde c(k,z)/c(k,z)\right|.$$

$\log_{10}\mathcal{E}=-3$ is a one-dollar mistake per \$1,000 of consumption.

In [ ]:
def euler_errors(model, cpol, k_test):
    k, z, Pi = model.k, model.z, model.Pi
    alpha, beta = model.alpha, model.beta
    E = np.zeros((k_test.size, z.size))
    for j in range(z.size):
        for i, kk in enumerate(k_test):
            cc = np.interp(kk, k, cpol[:, j])
            kp = z[j] * kk**alpha - cc
            rhs = sum(Pi[j, mm] * alpha * z[mm] * kp**(alpha - 1)
                      / np.interp(kp, k, cpol[:, mm]) for mm in range(z.size))
            E[i, j] = abs(1.0 - (1.0 / (beta * rhs)) / cc)
    return E


k_test = np.linspace(0.06, 0.59, 997)        # deliberately between grid points
E_vfi = euler_errors(ms, c_vfi, k_test)
E_ti = euler_errors(ms, c_ti, k_test)

plt.figure(figsize=(8, 4))
plt.plot(k_test, np.log10(E_vfi[:, 0]), lw=0.9, color="firebrick", label="VFI on a grid")
plt.plot(k_test, np.log10(E_ti[:, 0]), lw=0.9, color="darkorange", label="time iteration")
plt.axhline(-3, color="grey", ls="--", label='"acceptable" $=10^{-3}$')
plt.xlabel("$k$ (off the solution grid)"); plt.ylabel(r"$\log_{10}\mathcal{E}$")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

for name, E in [("VFI", E_vfi), ("time iteration", E_ti)]:
    print(f"{name:16s} mean log10 E = {np.log10(E.mean()):6.2f}   "
          f"worst = {np.log10(E.max()):6.2f}")

**Exercise 5.** Evaluate the residuals *on* the solution grid instead
(`k_test = ms.k[1:-1]`). The numbers get better. Explain why that makes them useless as a
measure of accuracy.

---
## Part 7 — Moments, and the filter that makes them comparable

Block B's last deck covers the data side of this: acquire, clean, persist, export. The one
step you need for **Homework 1 Part B** is the filter, so here it is on a series we generate
ourselves — no internet required, and no dependency beyond NumPy.

In [ ]:
def hp_filter(y, lamb=1600.0):
    """Hodrick-Prescott filter, written out in four lines of linear algebra.

    Minimises  sum_t (y_t - tau_t)^2 + lamb * sum_t (tau_{t+1} - 2 tau_t + tau_{t-1})^2,
    whose solution is  tau = (I + lamb * K'K)^{-1} y, with K the second-difference
    matrix.  Returns (cycle, trend), in statsmodels' order.
    """
    y = np.asarray(y, dtype=float)
    T = y.size
    K = np.zeros((T - 2, T))
    for i in range(T - 2):
        K[i, i:i+3] = [1.0, -2.0, 1.0]
    trend = np.linalg.solve(np.eye(T) + lamb * K.T @ K, y)
    return y - trend, trend


# A simulated log-output series: trend growth + an AR(1) cycle.
rng = np.random.default_rng(0)
T = 240                                   # 60 years, quarterly
eps = rng.normal(0, 0.007, T)
cyc = np.zeros(T)
for t in range(1, T):
    cyc[t] = 0.95 * cyc[t-1] + eps[t]
log_y = np.log(100.0) + 0.005 * np.arange(T) + cyc

cycle, trend = hp_filter(log_y, lamb=1600)   # 1600 = the quarterly convention

print(f"sd of the cyclical component   {cycle.std():.4f}")
print(f"first-order autocorrelation    {np.corrcoef(cycle[1:], cycle[:-1])[0,1]:.4f}")
print(f"(the AR(1) we actually drew had rho = 0.95, sd = {cyc.std():.4f})")

# statsmodels ships the same filter; cross-check ours when it is installed.
try:
    from statsmodels.tsa.filters.hp_filter import hpfilter
    _, t_sm = hpfilter(log_y, lamb=1600)
    print(f"\nmax |ours - statsmodels| = {np.max(np.abs(trend - t_sm)):.2e}")
except ImportError:
    print("\n(statsmodels not installed - skipping the cross-check)")

Two things to notice, both of which matter for the homework.

1. The filter does **not** return the process you drew. It returns whatever is left after a
   smooth trend is removed, and at `lamb=1600` some of a persistent cycle goes into the trend.
2. So the only fair comparison is **like with like**: filter the model's simulated series
   exactly as you filtered the data. That is the whole point of fixing `lamb` by convention.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(log_y, lw=1.0, label="log output", color="grey")
ax[0].plot(trend, lw=2.0, label="HP trend", color="firebrick")
ax[0].set_xlabel("quarter"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(cycle, lw=1.0, color="darkorange", label="HP cycle")
ax[1].axhline(0, ls="--", color="grey")
ax[1].set_xlabel("quarter"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Exercise 6.** Re-run with `lamb=100` and `lamb=129600`. Which one leaves almost everything
in the cycle, and which one leaves almost nothing? Now explain why the convention has to be
fixed *before* you compare a model to data.

---
## Part 8 — A first taste of PyTorch

Everything from Session 4 onwards needs derivatives. Here is why we will not compute them
by hand.

In [ ]:
import torch

k0, a = 2.0, 0.36
exact = a * k0**(a - 1)

print(f"{'h':>10}{'forward difference':>22}{'relative error':>18}")
for e in range(1, 17):
    h = 10.0**(-e)
    fd = ((k0 + h)**a - k0**a) / h
    print(f"{h:10.0e}{fd:22.15f}{abs(fd/exact - 1):18.2e}")

x = torch.tensor(k0, requires_grad=True, dtype=torch.float64)
y = x**a
y.backward()
print(f"\nautograd    {float(x.grad):22.15f}{abs(float(x.grad)/exact - 1):18.2e}")
print(f"exact       {exact:22.15f}")

Finite differences bottom out near $\sqrt{\varepsilon_{\text{mach}}}\approx1.5\times10^{-8}$
and then get **worse**: at $h=10^{-16}$ the numerator underflows to zero and the answer is
100% wrong. Automatic differentiation has no such floor — it applies the chain rule,
exactly.

---
## What to take away

1. Twenty lines take you from the Bellman equation to a policy. Knowing **how wrong** it is
   takes the rest of the lab.
2. The iteration count belongs to $\beta$; the accuracy belongs to the grid.
3. The quantity you monitor is a factor $\beta/(1-\beta)$ below the error you care about.
4. Error budgets compose — refining one past the others buys nothing.
5. Two methods resting on different theorems, agreeing to four digits, is the strongest
   evidence available when there is no closed form. Here there is one, so use it.
6. A model moment and a data moment are comparable only if they were filtered the same way.

**Homework 1, Part A** starts from this lab.